# Phishing Website Detection using Machine Learning

**Authors:** Chaymae Hanida, Aya El Hadri

This notebook compares 5 supervised classification models (Logistic Regression, KNN, Decision Tree, Random Forest, SVM) to automatically detect phishing websites, using a dataset of 11,054 websites (phishing / legitimate) described by 30 URL and domain-based features.

**Pipeline:** load data → preprocess & scale → train 5 models → evaluate (accuracy, precision, recall, F1-score) → compare → feature importance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import (
    DecisionTreeClassifier,
    plot_tree
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

## 1. Load the dataset

The dataset (`data/phishing.csv`) contains 11,054 websites labeled as phishing (`-1`) or legitimate (`1`), described by 30 features (e.g. `UsingIP`, `HTTPS`, `AnchorURL`, `AgeofDomain`).

In [ ]:
df = pd.read_csv("../data/phishing.csv")

print("\nDataset preview")
print(df.head())

print("\nShape")
print(df.shape)

print("\nInfo")
print(df.info())

## 2. Preprocessing

- Drop the `Index` column (not a predictive feature).
- Split features (`X`) and target (`y = class`).
- 80/20 train-test split, stratified on the target to keep class balance.
- Standardize features with `StandardScaler` for scale-sensitive models (Logistic Regression, KNN, SVM).

In [ ]:
if "Index" in df.columns:
    df.drop("Index", axis=1, inplace=True)

X = df.drop("class", axis=1)
y = df["class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 3. Evaluation helper

Computes accuracy, precision, recall, F1-score, a classification report, and a confusion matrix heatmap for a given model.

In [ ]:
def evaluate_model(model, Xtest, ytest):

    y_pred = model.predict(Xtest)

    acc = accuracy_score(ytest, y_pred)
    prec = precision_score(ytest, y_pred)
    rec = recall_score(ytest, y_pred)
    f1 = f1_score(ytest, y_pred)

    print("\nAccuracy :", round(acc, 4))
    print("Precision :", round(prec, 4))
    print("Recall :", round(rec, 4))
    print("F1 Score :", round(f1, 4))

    print("\nClassification Report\n")
    print(classification_report(ytest, y_pred))

    cm = confusion_matrix(ytest, y_pred)

    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(type(model).__name__)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

    return acc

results = {}

## 4. Linear Regression (baseline)

Not a classifier by design, but included as a naive baseline: predictions are thresholded at 0 to obtain a class label.

In [ ]:
print("\n========================")
print("LINEAR REGRESSION (baseline)")
print("========================")

linear = LinearRegression()
linear.fit(X_train_scaled, y_train)

pred = linear.predict(X_test_scaled)
pred_class = np.where(pred >= 0, 1, -1)

acc = accuracy_score(y_test, pred_class)
print("Accuracy :", round(acc, 4))

results["Linear Regression"] = acc

## 5. Logistic Regression

In [ ]:
print("LOGISTIC REGRESSION")
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_scaled, y_train)

results["Logistic Regression"] = evaluate_model(
    log_model,
    X_test_scaled,
    y_test
)

## 6. K-Nearest Neighbors (k=5)

In [ ]:
print("KNN")
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

results["KNN"] = evaluate_model(
    knn,
    X_test_scaled,
    y_test
)

## 7. Decision Tree (max_depth=5)

In [ ]:
print("DECISION TREE")
dt = DecisionTreeClassifier(random_state=42, max_depth=5)
dt.fit(X_train, y_train)

results["Decision Tree"] = evaluate_model(
    dt,
    X_test,
    y_test
)

plt.figure(figsize=(25, 12))
plot_tree(
    dt,
    feature_names=X.columns,
    class_names=["Phishing", "Legitimate"],
    filled=True,
    rounded=True,
    fontsize=8
)
plt.title("Decision Tree")
plt.show()

## 8. Random Forest (200 trees)

In [ ]:
print("RANDOM FOREST")
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

results["Random Forest"] = evaluate_model(
    rf,
    X_test,
    y_test
)

In [ ]:
# Visualize the first tree of the Random Forest
tree_1 = rf.estimators_[0]

plt.figure(figsize=(30, 15))
plot_tree(
    tree_1,
    feature_names=X.columns,
    class_names=["Phishing", "Legitimate"],
    filled=True,
    rounded=True,
    fontsize=7,
    max_depth=4
)
plt.title("First tree of the Random Forest")
plt.show()

## 9. SVM (RBF kernel)

In [ ]:
print("SVM")
svm = SVC(kernel='rbf')
svm.fit(X_train_scaled, y_train)

results["SVM"] = evaluate_model(
    svm,
    X_test_scaled,
    y_test
)

## 10. Model comparison

In [ ]:
result_df = pd.DataFrame(
    results.items(),
    columns=["Model", "Accuracy"]
)
print("\nFINAL RESULTS")
print(result_df)

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(
    data=result_df,
    x="Accuracy",
    y="Model"
)
plt.title("Model Comparison")
plt.show()

## 11. Feature importance (Random Forest)

The Random Forest model — the best performer — is used to rank the most discriminant features for detecting phishing sites.

In [ ]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

importance = importance.sort_values(by="Importance", ascending=False)

print("\nTop Features")
print(importance.head(15))

plt.figure(figsize=(10, 8))
sns.barplot(
    data=importance.head(15),
    x="Importance",
    y="Feature"
)
plt.title("Top 15 Most Important Features")
plt.show()

## 12. Save results

In [ ]:
import os
os.makedirs("../results", exist_ok=True)

result_df.to_csv("../results/model_accuracy_comparison.csv", index=False)

print("\nresults/model_accuracy_comparison.csv created.")

## Conclusion

| Model | Accuracy | Precision | Recall | F1-score | Rank |
|---|---|---|---|---|---|
| **Random Forest** | **97.38%** | **96.88%** | **98.46%** | **97.67%** | 1st |
| SVM (RBF kernel) | 96.02% | 95.40% | 97.56% | 96.47% | 2nd |
| KNN (k=5) | 94.08% | 94.44% | 95.05% | 94.74% | 3rd |
| Logistic Regression | 93.89% | 94.06% | 95.05% | 94.55% | 4th |
| Decision Tree (max_depth=5) | 93.35% | 91.76% | 96.75% | 94.19% | 5th |

**Random Forest** is the best-performing model, thanks to its ensemble of 200 trees voting by majority, which resists overfitting and captures non-linear relationships. The most discriminant features are **HTTPS** and **AnchorURL** — phishing sites typically avoid valid SSL certificates and use suspicious anchor links.